In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 00. Setup & Data Governance: Unity Catalog Architecture
# MAGIC 
# MAGIC - **Purpose**: Establish catalog, medallion schema boundaries (`bronze`, `silver`, `gold`), tags, and data masking governance.
# MAGIC - **Execution Environment**: Serverless Compute with Azure Databricks Unity Catalog.

# COMMAND ----------


In [0]:

# 1. Establish Medallion Schemas within Active Catalog
schemas = ["bronze", "silver", "gold"]

for schema in schemas:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema}")
    print(f"Verified schema: {schema}")

# COMMAND ----------

# 2. Add Formal Medallion Metadata Comments
spark.sql("COMMENT ON SCHEMA bronze IS 'Raw landing zone: Immutable 1:1 ingestion with audit metadata (_ingested_at, _source_file).'")
spark.sql("COMMENT ON SCHEMA silver IS 'Conformed analytical layer: Deduplicated, SCD Type 2 history tracking, and surrogate keys.'")
spark.sql("COMMENT ON SCHEMA gold IS 'Curated business aggregates: Star schema dimensional models, RFM cohorts, and executive KPIs.'")

print("Schema comments and architectural classifications applied.")

# COMMAND ----------


In [0]:

# 3. Governance Tags (Categorization for Data Lineage & Compliance)
# Tagging schemas with data classification and environment
spark.sql("ALTER SCHEMA bronze SET TAGS ('tier' = 'bronze', 'lifecycle' = 'ingestion', 'contains_raw_pii' = 'true')")
spark.sql("ALTER SCHEMA silver SET TAGS ('tier' = 'silver', 'lifecycle' = 'cleansed', 'scd_pattern' = 'type_2')")
spark.sql("ALTER SCHEMA gold SET TAGS ('tier' = 'gold', 'lifecycle' = 'curated', 'audience' = 'bi_and_dashboards')")

print("Governance metadata tags registered in Unity Catalog.")

# COMMAND ----------


In [0]:

# 4. Data Privacy & Governance: Dynamic Masking Demonstration (SQL Definition)
# In enterprise settings, raw user emails or addresses are masked for non-admin personas
spark.sql("""
CREATE OR REPLACE FUNCTION silver.mask_email(email STRING)
RETURNS STRING
RETURN CASE 
  WHEN is_account_group_member('data_engineers') THEN email
  ELSE CONCAT(LEFT(email, 2), '****@', SPLIT(email, '@')[1])
END
""")

print("Dynamic data masking function registered: silver.mask_email")

# COMMAND ----------

# 5. Display Catalog & Governance Summary
display(spark.sql("SHOW SCHEMAS"))